# Django Chatbot with Tool Use

Interactive chatbot that answers Django questions using OpenAI's tool calling feature with Gradio UI.

## Import modules and packages

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

## Initialize API Client

In [ ]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

## System Prompt Setup

In [ ]:
system_message = """
You are a helpful Django documentation assistant.
Explain Django concepts simply and practically.
If you are unsure, say so.
Keep answers beginner-friendly.
"""

## Define Available Tools

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_django_docs",
            "description": "Searches a small Django documentation knowledge base for a topic such as models, views, urls, templates, forms, or mvt.",
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {
                        "type": "string",
                        "description": "The Django topic to search for, for example models, views, urls, templates, forms, or mvt."
                    }
                },
                "required": ["topic"],
                "additionalProperties": False
            }
        }
    }
]

## Chat Logic with Tool Use

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_message}]
    
    for user_msg, assistant_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": assistant_msg})
    
    messages.append({"role": "user", "content": message})
    
    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools
    )
    
    response_message = response.choices[0].message
    
    if response_message.tool_calls:
        tool_call = response_message.tool_calls[0]
        
        print("Tool requested:")
        print(tool_call)
        
        arguments = json.loads(tool_call.function.arguments)
        
        topic = arguments.get("topic")
        tool_result = search_django_docs(topic)
        
        print("Tool result:")
        print(tool_result)
        
        messages.append(response_message)
        
        messages.append({
            "role":"tool",
            "tool_call_id":tool_call.id,
            "content": tool_result
        })
        
        second_response = openai.chat.completions.create(
            model=MODEL,
            messages=messages
        )
        
        return second_response.choices[0].message.content
    else:
        return response_message.content
    

## Knowledge Base & Helper Functions

In [ ]:
def search_django_docs(topic):
    print(f"search_django_docs() called with: {topic}")
    
    topic = topic.lower().strip()
    
    if topic in DJANGO_DOCS:
        return DJANGO_DOCS[topic]
    else:
        return "No matching Django documentation topic found."

In [ ]:
DJANGO_DOCS = {
    "models": """
Django models are Python classes that define the structure of your database tables.
Each model usually maps to one database table.
Each attribute on the model usually maps to one database column.

Example:
class Product(models.Model):
    name = models.CharField(max_length=100)
    price = models.DecimalField(max_digits=10, decimal_places=2)
""",

    "views": """
Django views contain the logic for handling a web request and returning a response.
A view receives a request and returns something like an HTML page, JSON response, or redirect.
""",

    "urls": """
Django URLs connect browser paths to view functions.
For example, the URL /products/ can be connected to a view called product_list.
""",

    "templates": """
Django templates are HTML files that can include dynamic data.
They are used to display information from your views to the user.
""",

    "forms": """
Django forms help you collect, validate, and process user input.
They are commonly used for login forms, contact forms, signup forms, and admin-style data entry.
""",

    "mvt": """
MVT stands for Model-View-Template.
Model handles data and database structure.
View handles request/response logic.
Template handles the HTML shown to the user.
"""
}

## Launch Chatbot Interface

In [ ]:
gr.ChatInterface(fn=chat).launch()